# Econ 521 — Lab 1
## Potential Outcomes, Selection Bias, and Randomization


Thursday's lecture gave you the Rubin Causal Model and the fundamental problem of causal
inference. Today we make those objects concrete: we will build potential outcomes in a
dataframe, watch a *simple difference in outcomes* fail, decompose exactly why it failed,
and then look at the same failure in real data from a real experiment.

**By the end of the hour you should be able to:**

1. Write down $Y_i^1$, $Y_i^0$, $\delta_i$, and the switching equation in code, and explain why
   the last column of a potential-outcomes table can never be observed.
2. Compute the ATE, ATT, ATU, and the SDO from a dataset where you (unrealistically) see both
   potential outcomes.
3. Decompose the SDO into the ATE, selection bias, and heterogeneous treatment effect bias,
   and verify the identity numerically.
4. Explain, using the Perfect Doctor / Bad Doctor contrast, why *good* decision-making by the
   people you study is the main source of bias in your data.
5. Show that selection bias does **not** go away as $n \to \infty$.


**Readings this lab leans on:** *Mastering 'Metrics* ch. 1; *Mixtape* ch. 4; *The Effect* ch. 10.

---
## Warm-up (5 min, no code)

Cunningham opens *Mixtape* ch. 4 with three stories. Take two minutes with the person next to
you and decide, for each one, **what the treatment assignment mechanism is** and **whether a
correlation would recover the causal effect**.

1. **The aliens and the ventilators.** Observers see that COVID patients placed on ventilators
   die more often than patients who are not. They remove the ventilators. Deaths rise.

2. **The rooster and the sunrise.** The rooster crows, then the sun comes up, every single day.
   One morning a cat kills the rooster. The sun comes up anyway.

3. **The sailor and the rudder.** A sailor moves the rudder constantly to counteract a
   crosswind, and the boat travels in a perfectly straight line. An observer sees the rudder
   moving and the heading not changing, and concludes the rudder is broken.

Three distinct failures: *sorting on treatment gains* (1), *post hoc ergo propter hoc* (2), and
*causality with no correlation left behind* (3). Number 3 is the one people forget: a skilled
optimizer can drive the observed covariance to zero while the causal effect is large. The Fed
responding to output gaps is the same story with a research budget attached.

The common thread: **when intentional actors choose treatment based on what they expect it to
do for them, correlations stop tracking causal effects.** Everything today is an elaboration of
that sentence.

In [37]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
rng = np.random.default_rng(20260904)

# causaldata ships the LaLonde and Thornton extracts we use later.
# If it is not installed:  pip install causaldata
from causaldata import nsw_mixtape, cps_mixtape

---
## §1 Potential outcomes in a dataframe

Ten COVID patients. $D_i = 1$ if patient $i$ is placed on a ventilator. The two potential
outcomes are **post-treatment years of life**:

$$Y_i^1 = \text{years of life if on a ventilator}, \qquad Y_i^0 = \text{years of life if not}.$$

The individual treatment effect is $\delta_i = Y_i^1 - Y_i^0$, and the realized outcome comes
from the **switching equation**

$$Y_i = D_i Y_i^1 + (1 - D_i) Y_i^0 .$$

The table below is a *God's-eye view*: it shows both potential outcomes for every patient. No
dataset you will ever analyze looks like this. That is the point of starting here — we can
compute the answer, then watch estimators try and fail to recover it.

In [38]:
patients = pd.DataFrame({
    "patient": range(1, 11),
    "y1": [7, 5, 5, 7, 4, 10, 1, 5, 3, 9],
    "y0": [1, 6, 1, 8, 2,  1, 10, 6, 7, 8],
})
patients["delta"] = patients["y1"] - patients["y0"]
patients

,patient,y1,y0,delta
0,1,7,1,6
1,2,5,6,-1
2,3,5,1,4
3,4,7,8,-1
4,5,4,2,2
5,6,10,1,9
6,7,1,10,-9
7,8,5,6,-1
8,9,3,7,-4
9,10,9,8,1


In [39]:
ATE = patients["delta"].mean()
print(f"E[Y1]  = {patients.y1.mean():.2f}")
print(f"E[Y0]  = {patients.y0.mean():.2f}")
print(f"ATE    = {ATE:.2f}   (averaging deltas and differencing the means agree by linearity)")

E[Y1]  = 5.60
E[Y0]  = 5.00
ATE    = 0.60   (averaging deltas and differencing the means agree by linearity)


Note what is *already* visible: treatment effects are wildly heterogeneous. Patient 6 gains 9
years; patient 7 loses 9. The ATE of $+0.6$ is a weighted average that hides both.

### Exercise 1 (5 min)

Enter the **Perfect Doctor**. She knows every patient's $\delta_i$ in advance and assigns the
ventilator to exactly those patients it helps. In the cell below:

1. Create `D` equal to 1 when `delta > 0` and 0 otherwise.
2. Use the switching equation to build the realized outcome `y`.
3. Compute the **ATT** ($E[\delta_i \mid D_i = 1]$), the **ATU** ($E[\delta_i \mid D_i = 0]$),
   and the **SDO**, $E[Y_i \mid D_i = 1] - E[Y_i \mid D_i = 0]$.

Before you run it, write down a prediction: will the SDO be bigger or smaller than $+0.6$?

In [40]:
# ---- EXERCISE 1 -------------------------------------------------------------
df = patients.copy()

# 1. Perfect Doctor's assignment rule
df["D"] = (df["delta"] > 0).astype(int)

# 2. Switching equation: Y = D*Y1 + (1-D)*Y0
df["y"] = df["D"] * df["y1"] + (1 - df["D"]) * df["y0"]

# 3. Causal parameters
ATT = df.loc[df["D"] == 1, "delta"].mean()   # mean of delta among the treated
ATU = df.loc[df["D"] == 0, "delta"].mean()   # mean of delta among the untreated
SDO = df.loc[df["D"] == 1, "y"].mean() - df.loc[df["D"] == 0, "y"].mean()

print(f"ATE = {ATE:.2f}\nATT = {ATT:.2f}\nATU = {ATU:.2f}\nSDO = {SDO:.2f}")
df
# -----------------------------------------------------------------------------

ATE = 0.60
ATT = 4.40
ATU = -3.20
SDO = -0.40


,patient,y1,y0,delta,D,y
0,1,7,1,6,1,7
1,2,5,6,-1,0,6
2,3,5,1,4,1,5
3,4,7,8,-1,0,8
4,5,4,2,2,1,4
5,6,10,1,9,1,10
6,7,1,10,-9,0,10
7,8,5,6,-1,0,6
8,9,3,7,-4,0,7
9,10,9,8,1,1,9


**Check yourself:** ATT $= +4.4$, ATU $= -3.2$, SDO $= -0.4$.

Sit with that for a second. The true average effect of the ventilator is **positive** ($+0.6$
years). The number a researcher would actually compute from this dataset is **negative**
($-0.4$ years). The estimator did not malfunction. The doctor was *good at her job*, and that
is what broke it.

Also confirm the weighting identity, with $\pi$ the share treated:

$$\text{ATE} = \pi \cdot \text{ATT} + (1-\pi)\cdot \text{ATU} = 0.5(4.4) + 0.5(-3.2) = 0.6 .$$

---
## §2 Decomposing the SDO

Where did the $-1.0$ discrepancy come from? Start from the definition of the ATE as a weighted
average of the ATT and ATU, add and subtract terms, and rearrange (the algebra is in *Mixtape*
§4.3 — do it once by hand this week). You land on an **identity**, not a theorem:

$$
\underbrace{E[Y \mid D=1] - E[Y \mid D=0]}_{\text{SDO}}
= \underbrace{E[Y^1] - E[Y^0]}_{\text{ATE}}
+ \underbrace{E[Y^0 \mid D=1] - E[Y^0 \mid D=0]}_{\text{selection bias}}
+ \underbrace{(1-\pi)\,(\text{ATT} - \text{ATU})}_{\text{heterogeneous TE bias}}
$$

Two things to notice about the middle term. First, it is a comparison of $Y^0$ across groups —
how the treated *would have* done untreated versus how the untreated actually did. Second, that
is precisely the quantity no dataset contains. Every research design in this course is a
strategy for arguing that this term is zero, or for constructing a comparison group that makes
it zero.

The third term is unfamiliar to most people. It is zero whenever the treated and untreated
benefit equally on average. Under the Perfect Doctor it cannot be zero, because she sorted on
the gain itself.

In [41]:
def decompose(data, dcol="D", y1="y1", y0="y0"):
    '''Return the SDO and its three components. Requires BOTH potential outcomes.'''
    d = data[dcol].astype(bool)
    pi = d.mean()
    ate = (data[y1] - data[y0]).mean()
    att = (data.loc[d, y1] - data.loc[d, y0]).mean()
    atu = (data.loc[~d, y1] - data.loc[~d, y0]).mean()
    selection = data.loc[d, y0].mean() - data.loc[~d, y0].mean()
    het = (1 - pi) * (att - atu)
    y = np.where(d, data[y1], data[y0])
    sdo = y[d].mean() - y[~d].mean()
    return pd.Series({"pi": pi, "ATE": ate, "ATT": att, "ATU": atu,
                      "selection bias": selection, "het. TE bias": het,
                      "SDO (observed)": sdo, "ATE+sel+het": ate + selection + het})

decompose(df)

pi                0.500
ATE               0.600
ATT               4.400
ATU              -3.200
selection bias   -4.800
het. TE bias      3.800
SDO (observed)   -0.400
ATE+sel+het      -0.400
dtype: float64

The last two rows agree to machine precision, because this is an identity. The SDO of $-0.4$ is
$0.6$ of signal, $-4.8$ of selection bias, and $+3.8$ of heterogeneity bias.

The selection bias term of $-4.8$ has a plain-English reading: **had the ventilator patients not
been ventilated, they would have lived 4.8 fewer years than the patients who genuinely were not
ventilated.** They were sicker. Everyone knows this about hospitals; almost nobody knows the
equivalent fact about the settings we actually study, which is why we need designs.

---
## §3 Perfect vs. Bad Doctor at scale

Now meet the **Bad Doctor**. He does not look at potential outcomes at all — the first half of
the patients he ever sees go on ventilators, the rest go home. By any clinical standard he is
worse. By our standard he is a gift, because his assignment rule ignores $(Y^0, Y^1)$, which is
exactly the independence condition

$$(Y^1, Y^0) \perp\!\!\!\perp D .$$

Same 100,000 patients, same potential outcomes, two assignment rules. Only the *mechanism*
differs.

In [42]:
n = 100_000
sim = pd.DataFrame({
    "y1": rng.normal(10.0, 4.0, n).clip(min=0),   # lifespan on a vent
    "y0": rng.normal( 9.4, 4.0, n).clip(min=0),   # lifespan off a vent
})
sim["delta"] = sim.y1 - sim.y0

sim["D_perfect"] = (sim.delta > 0).astype(int)          # sorts on the gain
sim["D_bad"]     = (np.arange(n) < n // 2).astype(int)  # ignores the gain entirely

comparison = pd.DataFrame({
    "Perfect Doctor": decompose(sim, "D_perfect"),
    "Bad Doctor":     decompose(sim, "D_bad"),
})
comparison.round(3)

,Perfect Doctor,Bad Doctor
pi,0.542,0.500
ATE,0.612,0.612
ATT,4.720,0.595
ATU,-4.256,0.628
selection bias,-4.485,0.020
het. TE bias,4.109,-0.016
SDO (observed),0.235,0.615
ATE+sel+het,0.235,0.615


Read the columns against each other.

- **ATE is identical** in both — of course it is. The ATE is a property of the patients, not of
  who assigns them. Assignment is the only thing the doctors do.
- Under the Perfect Doctor, ATT $\gg$ ATE $\gg$ ATU, selection bias is large and negative, and
  the SDO misses badly.
- Under the Bad Doctor, ATE $\approx$ ATT $\approx$ ATU, selection bias is a rounding error, and
  **SDO $\approx$ ATE**.

This is the whole argument for randomization, and notice how it was made: not by appealing to
"balancing covariates," but by showing that independent assignment sets the two bias terms to
zero. Randomization balances observed covariates, unobserved covariates, potential outcomes,
*and the treatment effects themselves*. Covariate balance is a testable symptom, not the
mechanism.

### Exercise 2 (4 min) — does bias shrink with sample size?

A common instinct is that bias is a small-sample problem. Test it. For each $n$ in
`[250, 1_000, 10_000, 100_000]`, run 200 replications: redraw the potential outcomes, apply the
Perfect Doctor rule, and record the SDO. Then report, for each $n$, the **mean SDO**, the
**standard deviation of the SDO** across replications, and the **mean gap** (SDO minus the true
ATE of about 0.61).

One of those three columns should collapse toward zero as $n$ grows. Predict which one first.

In [43]:
# ---- EXERCISE 2 -------------------------------------------------------------
def one_draw(n_i):
    '''Draw n_i patients, apply the Perfect Doctor rule, return (ATE, SDO).'''
    y1 = rng.normal(10.0, 4.0, n_i).clip(min=0)
    y0 = rng.normal( 9.4, 4.0, n_i).clip(min=0)
    d  = (y1 - y0) > 0              # Perfect Doctor rule (boolean array)
    y  = np.where(d, y1, y0)        # switching equation
    ate = (y1 - y0).mean()
    sdo = y[d].mean() - y[~d].mean()  # mean of y among treated minus mean among untreated
    return ate, sdo

rows = []
for n_i in [250, 1_000, 10_000, 100_000]:
    draws = np.array([one_draw(n_i) for _ in range(200)])
    ate_bar, sdo_bar = draws.mean(axis=0)
    rows.append({"n": n_i,
                 "mean SDO": sdo_bar,
                 "sd of SDO": draws[:, 1].std(),
                 "mean gap (SDO - ATE)": sdo_bar - ate_bar})

pd.DataFrame(rows).round(4)
# -----------------------------------------------------------------------------


,n,mean SDO,sd of SDO,mean gap (SDO - ATE)
0,250,0.153,0.442,-0.398
1,1000,0.242,0.198,-0.358
2,10000,0.221,0.069,-0.371
3,100000,0.219,0.022,-0.374


**What you should see:** the standard deviation of the SDO falls by roughly a factor of two
every time $n$ quadruples — that is the familiar $O(n^{-1/2})$ — while the mean gap sits stubbornly
around $-0.37$ at every sample size.

Sampling noise is $O(n^{-1/2})$. Bias is $O(1)$. More data buys you a tighter confidence interval
around the wrong number, and at large $n$ it buys you a *confidently* wrong number, since the
standard error shrinks while the bias does not. Worth remembering the next time someone waves ten
million rows at you.

---
## §4 Real data: LaLonde (1986) 

* LaLonde, R. J. (1986). Evaluating the econometric evaluations of training programs with experimental data. The American economic review, 604-620.


Simulations are persuasive to people who already believe you. Here is the same phenomenon in
data, from the paper that reorganized how economists think about non-experimental evaluation.

**Setting.** The National Supported Work Demonstration (NSW) was a mid-1970s job training
program that randomly assigned disadvantaged applicants — ex-offenders, former drug users,
long-term welfare recipients — to a subsidized work experience treatment or to a control group.
Because assignment was random, the experimental treatment/control comparison estimates the ATE
directly.

**LaLonde's move.** LaLonde (1986, *AER*) threw away the experimental controls and replaced them
with an observational comparison group drawn from the CPS, then asked whether standard
econometric methods could recover the experimental answer. They could not, and the paper is why
"we ran a regression with controls" stopped being an acceptable identification strategy.

We use the Dehejia–Wahba extract. Outcome: **`re78`**, real earnings in 1978 (dollars).

In [44]:
nsw = nsw_mixtape.load_pandas().data       # experimental treated + experimental controls
cps = cps_mixtape.load_pandas().data       # CPS comparison group (never in the experiment)

print(nsw.shape, cps.shape)
print(nsw.groupby("treat").size().rename("n").to_frame().T)
nsw.head()

(445, 11) (15992, 11)
treat    0    1
n      260  185


,data_id,treat,age,educ,black,hisp,marr,nodegree,re74,re75,re78
0,Dehejia-Wahba Sample,1,37,11,1,0,1,1,0.000,0.000,"9,930.046"
1,Dehejia-Wahba Sample,1,22,9,0,1,0,1,0.000,0.000,"3,595.894"
2,Dehejia-Wahba Sample,1,30,12,1,0,0,0,0.000,0.000,"24,909.449"
3,Dehejia-Wahba Sample,1,27,11,1,0,0,1,0.000,0.000,"7,506.146"
4,Dehejia-Wahba Sample,1,33,8,1,0,0,1,0.000,0.000,289.790


In [45]:
exp_means = nsw.groupby("treat")["re78"].agg(["mean", "std", "count"])
exp_ate   = exp_means.loc[1, "mean"] - exp_means.loc[0, "mean"]

print(exp_means.round(2).to_string())
print(f"\nExperimental estimate of the ATE: ${exp_ate:,.0f} in 1978 earnings")

# identical number, with a standard error attached
smf.ols("re78 ~ treat", data=nsw).fit(cov_type="HC1").summary().tables[1]

           mean       std  count
treat                           
0     4,554.800 5,483.840    260
1     6,349.140 7,867.400    185

Experimental estimate of the ATE: $1,794 in 1978 earnings


,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,4554.8011,340.204,13.388,0.000,3888.014,5221.588
treat,1794.3424,670.824,2.675,0.007,479.551,3109.134


About **\$1,794**, and the difference in means and the OLS slope are the same number by
construction — a regression on a single binary regressor *is* a difference in means.

Now the observational version: keep the experimental **treated**, discard the experimental
controls, and compare them to the CPS instead. This is what an analyst without an experiment
would be forced to do.

In [46]:
obs = pd.concat([nsw.query("treat == 1"),
                 cps.assign(treat=0)], ignore_index=True)

obs_diff = obs.query("treat == 1").re78.mean() - obs.query("treat == 0").re78.mean()
print(f"n = {len(obs):,}")
print(f"Observational difference in means: ${obs_diff:,.0f}")
print(f"Experimental benchmark:            ${exp_ate:,.0f}")

n = 16,177
Observational difference in means: $-8,498
Experimental benchmark:            $1,794


The observational comparison says the training program **cost participants about \$8,500 a
year**. The experiment says it **raised earnings by about \$1,800**. Wrong magnitude, wrong
sign, and a sample size of 16,177 did nothing to help.

### Exercise 3 (5 min)

Diagnose it. Build a balance table comparing the treated group to (a) the experimental controls
and (b) the CPS controls, on `age`, `educ`, `black`, `hisp`, `marr`, `nodegree`, `re74`, `re75`.

Then answer in a sentence: which term of the §2 decomposition are you looking at when you
compare `re74` and `re75` across groups, and why are those two variables special here?

In [47]:
# ---- EXERCISE 3 -------------------------------------------------------------
covars = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]

exp_balance = nsw.groupby("treat")[covars].mean().T   # nsw grouped by treat, mean of covars
obs_balance = obs.groupby("treat")[covars].mean().T   # obs grouped by treat, mean of covars

# Suggested display: put them side by side
balance = pd.concat({"Experimental": exp_balance, "Observational": obs_balance}, axis=1)
balance.round(2)
# -----------------------------------------------------------------------------


Experimental           Observational          
treat               0         1             0         1
age            25.050    25.820        33.230    25.820
educ           10.090    10.350        12.030    10.350
black           0.830     0.840         0.070     0.840
hisp            0.110     0.060         0.070     0.060
marr            0.150     0.190         0.710     0.190
nodegree        0.830     0.710         0.300     0.710
re74        2,107.030 2,095.570    14,016.800 2,095.570
re75        1,266.910 1,532.060    13,650.800 1,532.060

**What the table shows.** In the experimental sample the two groups look alike on everything.
In the observational sample the NSW treated are about 7 years younger, 1.7 years less educated,
far more likely to be Black, far less likely to be married, and — the killer — they earned
roughly **\$12,000 less in 1974 and 1975**, *before the program existed*.

`re74` and `re75` are special because they are pre-treatment realizations of the outcome
variable. A pre-treatment gap in $Y$ is about as close as observational data ever gets to
letting you *see* the selection bias term $E[Y^0 \mid D=1] - E[Y^0 \mid D=0]$, and here it is
enormous and negative. The NSW recruited people precisely because they were doing badly. The
CPS did not.

If you have time later, try adding those controls to the observational regression and watch how
far short they fall. That failure is the content of LaLonde's paper, and it is what motivates
matching and propensity scores (12 November) — and what limits them.

---
## §5 What randomization does and does not promise (5 min)

One last caution, because it causes a lot of confusion in referee reports and in seminars.

Independence is a statement about **expectations in the population**. In any single finite
sample, the treatment and control groups will differ somewhat on everything, including things
you can see. That is not a failed randomization; it is sampling variation. Run the cell to see
how often a *correctly randomized* experiment produces a "significant" imbalance on a covariate
that has nothing to do with anything.

In [48]:
def one_experiment(n=200, n_covars=10):
    '''Randomize n units, test balance on pure-noise covariates. Return p-values.'''
    X = rng.normal(size=(n, n_covars))
    T = rng.binomial(1, 0.5, n).astype(bool)
    return np.array([
        smf.ols("x ~ T", data=pd.DataFrame({"x": X[:, j], "T": T.astype(int)}))
           .fit().pvalues["T"]
        for j in range(n_covars)
    ])

p = np.concatenate([one_experiment() for _ in range(200)])
print(f"share of balance tests with p < 0.05: {(p < 0.05).mean():.3f}")
print(f"share of experiments with >=1 'imbalance' out of 10 covariates: "
      f"{np.mean([one_experiment().min() < 0.05 for _ in range(200)]):.3f}")

share of balance tests with p < 0.05: 0.058
share of experiments with >=1 'imbalance' out of 10 covariates: 0.435


Roughly 5% of individual tests reject, by construction — and about **40%** of experiments show at
least one "imbalanced" covariate when you check ten of them. A balance table with one starred
row is not evidence of a broken experiment. It is evidence that you ran ten tests. We will come
back to this on 10 September when we talk about multiple testing and pre-analysis plans.

---

## Takeaways

1. The fundamental problem of causal inference is a missing-data problem where the missing data
   *does not exist*. No sample size fixes it.
2. Every observed difference in means decomposes into ATE + selection bias + heterogeneous
   treatment effect bias. Identification is the argument that the last two terms vanish.
3. The source of bias is usually not sloppiness. It is optimizing behavior by the people in your
   data — the Perfect Doctor, the eBay shopper who was going to buy anyway, the NSW applicant
   who enrolled because their earnings had collapsed.
4. Randomization eliminates both bias terms because it makes assignment independent of *all*
   potential outcomes, seen and unseen.
5. Bias does not shrink with $n$. Precision does.

